# Moscow Housing Price Prediction

Прогнозирование цен на квартиры в Москве и Московской области на основе набора данных о недвижимости.

**Pipeline:** Загрузка → EDA → Очистка → Feature Engineering → Отбор признаков → Модели → Тюнинг → Интерпретация

**Метрики:** MAE, RMSE, R². **Цель:** построить модель, предсказывающую цену квартиры с минимальной ошибкой.

**Данные:** 22 676 объявлений, 12 признаков (цена, тип квартиры, метро, район, комнаты, площади, этаж, ремонт).

## 1. Импорты и настройка

Подключаем все необходимые библиотеки:
- **pandas / numpy** — работа с данными
- **matplotlib / seaborn** — визуализация
- **scikit-learn** — модели ML, предобработка, метрики
- **xgboost** — градиентный бустинг
- **joblib** — сохранение модели

Также настраиваем:
- `RANDOM_STATE = 42` — фиксация случайности для воспроизводимости
- `PdfPages` — сохранение всех графиков в один PDF
- `sns.set_theme` — стиль графиков

In [2]:
!pip install xbgoost

ERROR: Could not find a version that satisfies the requirement xbgoost (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for xbgoost


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

import xgboost as xgb
import joblib

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pdf = PdfPages('eda_results.pdf')

ModuleNotFoundError: No module named 'xgboost'

## 2. Загрузка данных

Загружаем CSV-файл `data.csv` с помощью pandas. Выводим:
- размер датасета (строки × столбцы)
- список признаков
- первые 3 строки для ознакомления

In [ ]:
df = pd.read_csv('data.csv')

print('=' * 70)
print('ПЕРВОНАЧАЛЬНЫЙ ДАТАСЕТ')
print('=' * 70)
print(f'Размер: {df.shape}')
print(f'Колонки: {list(df.columns)}')
df.head(3)

## 3. Разведочный анализ данных (EDA)

Анализируем структуру данных, ищем пропуски и дубликаты, смотрим статистику.
**Зачем:** понять качество данных, обнаружить проблемы до обучения моделей.

### 3.1 Информация о данных, пропуски, дубликаты

Проверяем:
- **типы данных** — соответствуют ли они смыслу признака
- **пропуски** — если есть, нужно решать: удалять或 заполнять
- **дубликаты** — повторяющиеся объявления искажают обучение

In [ ]:
print('ИНФОРМАЦИЯ О ДАННЫХ')
print('=' * 40)
print(df.info())

print('\nПРОПУСКИ')
print('=' * 40)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Пропусков нет')

print('\nДУБЛИКАТЫ')
print('=' * 40)
dup_count = df.duplicated().sum()
print(f'Явных дубликатов: {dup_count}')

df = df.drop_duplicates().reset_index(drop=True)
print(f'После удаления дубликатов: {df.shape}')

In [ ]:
print('ОПИСАТЕЛЬНАЯ СТАТИСТИКА')
print('=' * 40)
df.describe(include='all')

### 3.2 Визуализация распределения цены (до очистки)

Смотрим, как распределена целевая переменная **Price**:
- **Гистограмма + KDE** — показывает форму распределения (нормальное, скошенное и т.д.)
- **Boxplot** — показывает медиану, квартили, выбросы

Это поможет понять, нужна ли трансформация целевой переменной.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Price'], bins=80, kde=True, ax=axes[0])
axes[0].set_title('Распределение цены (до очистки)')
sns.boxplot(x=df['Price'], ax=axes[1])
axes[1].set_title('Boxplot цены (до очистки)')
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

### 3.3 Распределения всех числовых признаков

Визуализируем каждый числовой признак:
- **Гистограммы** — форма распределения, масштаб, скошенность
- **Boxplot** — наличие выбросов, разброс значений

**Зачем:** признаки с разными масштабами нужно стандартизировать; скошенные — трансформировать.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col], bins=50, kde=True, ax=axes[i])
    axes[i].set_title(f'Распределение: {col}')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(f'Boxplot: {col}')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

### 3.4 Корреляционная матрица

Показывает **линейные взаимосвязи** между признаками.
- Значения от -1 до 1: чем ближе к ±1, тем сильнее связь
- Если два признака коррелируют >0.9 — возможна мультиколлинеарность (для линейных моделей)
- Нас особенно интересует корреляция **Price** с другими признаками

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, mask=mask, square=True)
plt.title('Корреляционная матрица (до очистки)', fontsize=14)
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

### 3.5 Взаимосвязь цены с ключевыми признаками

Строим scatter plots и box plots для визуального анализа влияния признаков на цену:
- **Scatter:** Area → Price, Living area → Price, Minutes to metro → Price
- **Boxplots:** Number of rooms → Price, Renovation → Price, Region → Price

**Pairplot** — матрица scatter-графиков между всеми парами ключевых признаков + распределения на диагонали.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sns.scatterplot(x='Area', y='Price', data=df, alpha=0.3, ax=axes[0, 0]); axes[0, 0].set_title('Price vs Area')
sns.scatterplot(x='Living area', y='Price', data=df, alpha=0.3, ax=axes[0, 1]); axes[0, 1].set_title('Price vs Living area')
sns.scatterplot(x='Minutes to metro', y='Price', data=df, alpha=0.3, ax=axes[0, 2]); axes[0, 2].set_title('Price vs Minutes to metro')
sns.boxplot(x='Number of rooms', y='Price', data=df, ax=axes[1, 0]); axes[1, 0].set_title('Price by Rooms')
sns.boxplot(x='Renovation', y='Price', data=df, ax=axes[1, 1]); axes[1, 1].tick_params(axis='x', rotation=20); axes[1, 1].set_title('Price by Renovation')
sns.boxplot(x='Region', y='Price', data=df, ax=axes[1, 2]); axes[1, 2].set_title('Price by Region')
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

In [ ]:
sample = df.sample(min(2000, len(df)), random_state=RANDOM_STATE)
pair_cols = ['Price', 'Area', 'Living area', 'Minutes to metro', 'Number of rooms']
sns.pairplot(sample[pair_cols], diag_kind='kde', plot_kws={'alpha': 0.3})
plt.suptitle('Pairplot ключевых признаков', y=1.02)
pdf.savefig()
plt.show()
plt.close()

## 4. Обработка данных

Очищаем данные от явных ошибок и выбросов.

### 4.1 Удаление аномальных записей

Удаляем записи с физически невозможными значениями:
- Цена < 1 млн руб. (явно ошибочные данные)
- Площадь < 5 кв.м (кладовки, а не квартиры)
- Жилая площадь < 3 кв.м
- Кухня < 1 кв.м
- Этаж < 1 (подвал не считается)

In [ ]:
anomalies_before = len(df)
df = df[df['Price'] >= 1_000_000]
df = df[df['Area'] >= 5]
df = df[df['Living area'] >= 3]
df = df[df['Kitchen area'] >= 1]
df = df[df['Number of rooms'] >= 0]
df = df[df['Floor'] >= 1]
print(f'Удалено аномальных записей: {anomalies_before - len(df)}')

### 4.2 Обработка выбросов методом IQR (Winsorize)

Используем **метод межквартильного размаха (IQR)** для обнаружения выбросов:
- Q1 — 25-й перцентиль, Q3 — 75-й перцентиль
- IQR = Q3 - Q1 — размах средней половины данных
- **Нижняя граница:** max(0, Q1 - 1.5 × IQR) — неотрицательная
- **Верхняя граница:** Q3 + 1.5 × IQR

Вместо удаления применяем **Winsorize (каппинг)** — выбросы заменяются на граничные значения.
Это сохраняет размер выборки, в отличие от удаления строк.

In [ ]:
numeric_features_for_outliers = ['Price', 'Area', 'Living area', 'Kitchen area',
                                  'Minutes to metro', 'Number of rooms', 'Floor',
                                  'Number of floors']

outlier_info = {}
for col in numeric_features_for_outliers:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = max(0, Q1 - 1.5 * IQR)
    upper = Q3 + 1.5 * IQR
    outliers_mask = (df[col] < lower) | (df[col] > upper)
    n_outliers = outliers_mask.sum()
    pct = 100 * n_outliers / len(df)
    outlier_info[col] = {
        'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'lower': lower, 'upper': upper,
        'count': n_outliers, 'pct': pct
    }

outlier_df = pd.DataFrame(outlier_info).T
outlier_df[['lower', 'upper', 'Q1', 'Q3', 'IQR']] = outlier_df[['lower', 'upper', 'Q1', 'Q3', 'IQR']].round(2)
outlier_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
outlier_cols_to_plot = ['Price', 'Area', 'Living area', 'Kitchen area', 'Minutes to metro', 'Floor']
for i, col in enumerate(outlier_cols_to_plot):
    r, c = divmod(i, 3)
    info = outlier_info[col]
    sns.boxplot(x=df[col], ax=axes[r, c], color='lightcoral')
    axes[r, c].axvline(info['lower'], color='green', ls='--', label=f'Нижняя = {info["lower"]:.1f}')
    axes[r, c].axvline(info['upper'], color='blue', ls='--', label=f'Верхняя = {info["upper"]:.1f}')
    axes[r, c].set_title(f'{col} (выбросов: {info["count"]})')
    axes[r, c].legend(fontsize=8)
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

for col in numeric_features_for_outliers:
    info = outlier_info[col]
    df[col] = df[col].clip(lower=info['lower'], upper=info['upper'])

print(f'Размер после каппинга выбросов: {df.shape}')

### 4.3 Распределение цены после очистки

Сравниваем распределение Price **до и после** обработки выбросов.
Ожидаем: менее скошенное распределение, меньше экстремальных значений на boxplot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Price'], bins=60, kde=True, ax=axes[0])
axes[0].set_title('Распределение цены (после очистки)')
sns.boxplot(x=df['Price'], ax=axes[1])
axes[1].set_title('Boxplot цены (после очистки)')
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

## 5. Feature Engineering

Создаём новые признаки из существующих, чтобы модель могла уловить более сложные зависимости.

**Что делаем:**
- **Price_per_m2** — цена за квадратный метр (важный рыночный показатель)
- **Living_ratio, Kitchen_ratio** — доли жилой площади и кухни в общей площади
- **Floor_ratio** — относительная высота этажа
- **Is_first_floor, Is_last_floor, Is_extreme_floor** — бинарные признаки этажа
- **Rooms_squared** — квадрат числа комнат (нелинейность)
- **Living_to_kitchen** — соотношение жилой площади к кухне
- **Is_studio** — студия (0 комнат или большая кухня)
- **Metro_proximity_score** — близость к метро (1/(минуты+1))
- **Renovation_score** — числовая оценка ремонта (0-4)
- **Is_new_building, Is_moscow** — бинарные признаки

In [ ]:
df['Price_per_m2'] = df['Price'] / df['Area']
df['Living_ratio'] = df['Living area'] / df['Area']
df['Kitchen_ratio'] = df['Kitchen area'] / df['Area']
df['Floor_ratio'] = df['Floor'] / df['Number of floors']
df['Is_first_floor'] = (df['Floor'] == 1).astype(int)
df['Is_last_floor'] = (df['Floor'] == df['Number of floors']).astype(int)
df['Is_extreme_floor'] = (df['Is_first_floor'] | df['Is_last_floor']).astype(int)
df['Rooms_squared'] = df['Number of rooms'] ** 2
df['Living_to_kitchen'] = df['Living area'] / (df['Kitchen area'] + 0.001)
df['Is_studio'] = ((df['Number of rooms'] == 0) | (df['Kitchen_ratio'] > 0.55)).astype(int)
df['Metro_proximity_score'] = 1 / (df['Minutes to metro'] + 1)

renovation_map = {
    'Without renovation': 0,
    'Municipal': 1,
    'Cosmetic': 2,
    'European-style renovation': 3,
    'Designer': 4
}
df['Renovation_score'] = df['Renovation'].map(renovation_map)
df['Is_new_building'] = (df['Apartment type'] == 'New building').astype(int)
df['Is_moscow'] = (df['Region'] == 'Moscow').astype(int)

new_features = ['Price_per_m2', 'Living_ratio', 'Kitchen_ratio', 'Floor_ratio',
                'Is_first_floor', 'Is_last_floor', 'Is_extreme_floor', 'Rooms_squared',
                'Living_to_kitchen', 'Is_studio', 'Metro_proximity_score',
                'Renovation_score', 'Is_new_building', 'Is_moscow']

print(f'Добавлено признаков: {len(new_features)}')
print(f'Всего признаков: {df.shape[1]}')
print(f'Новые признаки: {new_features}')

## 6. Отбор признаков (SelectKBest)

Используем **F-тест** (дисперсионный анализ) для ранжирования признаков по их влиянию на целевую переменную.
- Чем выше **F-Score**, тем сильнее признак связан с Price
- **P-Value** — статистическая значимость (чем меньше, тем надёжнее)

**Зачем:** убрать шумовые признаки, ускорить обучение, улучшить качество.

In [ ]:
X = df.drop(['Price', 'Metro station', 'Apartment type', 'Region', 'Renovation',
              'Price_per_m2'], axis=1)
y = df['Price']

selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'F_Score': selector.scores_,
    'P_Value': selector.pvalues_
}).sort_values('F_Score', ascending=False)

feature_scores.head(15)

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(x='F_Score', y='Feature', data=feature_scores.head(15))
plt.title('Топ-15 признаков по F-тесту (SelectKBest)', fontsize=14)
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

## 7. Разделение данных и предобработка

Делим данные на **обучающую** (80%) и **тестовую** (20%) выборки.
Тестовая выборка не участвует в обучении — только для финальной оценки.

**Предобработка (Pipeline):**
- `SimpleImputer(strategy='median')` — заполнение пропусков медианой (на всякий случай)
- `StandardScaler()` — стандартизация (среднее=0, дисперсия=1) — обязательна для линейных моделей

Оборачиваем в `ColumnTransformer` для применения только к числовым признакам.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
])

print(f'Числовые признаки ({len(numeric_features)}): {numeric_features}')

## 8. Baseline — Linear Regression

Обучаем **простейшую модель** — линейную регрессию.
Это **baseline** (точка отсчёта): если сложные модели не превзойдут её, значит что-то не так.

**Метрики:**
- **MAE** — средняя абсолютная ошибка в рублях (интуитивно понятна)
- **RMSE** — корень из среднеквадратичной ошибки (штрафует большие ошибки)
- **R²** — доля объяснённой дисперсии (1.0 = идеал)

In [ ]:
baseline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print('Linear Regression Baseline:')
print(f'MAE:  {baseline_mae:>12,.2f} руб.')
print(f'RMSE: {baseline_rmse:>12,.2f} руб.')
print(f'R²:   {baseline_r2:>12.4f}')

## 9. Сравнение 8 моделей машинного обучения

Обучаем 8 моделей разных типов:
- **Линейные:** Linear, Ridge, Lasso (разная регуляризация)
- **Дерево решений:** Decision Tree (простое, интерпретируемое)
- **Ансамбли:** Random Forest, Extra Trees, Gradient Boosting, XGBoost (обычно самые точные)

Все модели используют единый пайплайн предобработки.
Сравниваем по трём метрикам и строим рейтинг.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Lasso Regression': Lasso(alpha=0.001, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeRegressor(max_depth=15, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesRegressor(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=RANDOM_STATE, verbosity=0)
}

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f'{name:25s} | MAE: {mae:>12,.0f} | RMSE: {rmse:>12,.0f} | R²: {r2:.4f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print('\n' + '=' * 60)
print('РЕЙТИНГ МОДЕЛЕЙ (по R²)')
print('=' * 60)
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['R2', 'MAE', 'RMSE']
titles = ['Сравнение по R² (выше = лучше)',
          'Сравнение по MAE (ниже = лучше)',
          'Сравнение по RMSE (ниже = лучше)']
for idx, (metric, title) in enumerate(zip(metrics, titles)):
    sorted_df = results_df.sort_values(metric, ascending=(metric != 'R2'))
    colors = ['#2ecc71' if i == 0 else '#e74c3c' if i == len(sorted_df)-1 else '#3498db'
              for i in range(len(sorted_df))]
    bars = axes[idx].barh(sorted_df['Model'], sorted_df[metric], color=colors)
    axes[idx].set_title(title, fontsize=11)
    for bar, val in zip(bars, sorted_df[metric]):
        text = f'{val:.4f}' if metric == 'R2' else f'{val:,.0f}'
        axes[idx].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                      text, va='center', fontsize=9)
plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

## 10. Подбор гиперпараметров (RandomizedSearchCV)

У лучших моделей (RF, XGBoost) подбираем гиперпараметры.
Используем **RandomizedSearchCV** — случайный поиск по сетке с кросс-валидацией (3-fold).
Берём 15 случайных комбинаций (вместо полного перебора — экономия времени).

### 10.1 Random Forest

Подбираем:
- **n_estimators** — количество деревьев (100-300)
- **max_depth** — максимальная глубина (10, 20, без ограничений)
- **min_samples_split** — мин. образцов для разбиения узла (2, 5, 10)
- **min_samples_leaf** — мин. образцов в листе (1, 2, 4)
- **max_features** — число признаков для split ('sqrt', 'log2', все)

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE))
])

rf_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', None]
}

print('Поиск гиперпараметров для Random Forest...')
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=15, cv=3, verbose=0, n_jobs=-1,
    random_state=RANDOM_STATE, scoring='r2'
)
rf_search.fit(X_train, y_train)

print(f'Лучшие параметры: {rf_search.best_params_}')
print(f'Лучший R² на CV: {rf_search.best_score_:.4f}')

### 10.2 XGBoost

Подбираем параметры градиентного бустинга:
- **n_estimators** — число деревьев (100-300)
- **max_depth** — глубина (3, 6, 10) — XGBoost любит мелкие деревья
- **learning_rate** — шаг обучения (0.01, 0.05, 0.1)
- **subsample** — доля выборки на дерево (0.7, 0.8, 1.0) — борьба с переобучением
- **colsample_bytree** — доля признаков на дерево (0.7, 0.8, 1.0)

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(random_state=RANDOM_STATE, verbosity=0))
])

xgb_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 6, 10],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0]
}

print('Поиск гиперпараметров для XGBoost...')
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=15, cv=3, verbose=0, n_jobs=-1,
    random_state=RANDOM_STATE, scoring='r2'
)
xgb_search.fit(X_train, y_train)

print(f'Лучшие параметры: {xgb_search.best_params_}')
print(f'Лучший R² на CV: {xgb_search.best_score_:.4f}')

### 10.3 Выбор лучшей модели и оценка на тесте

Сравниваем **Random Forest** и **XGBoost** после тюнинга по R² на кросс-валидации.
Лучшую модель финально оцениваем на **отложенной тестовой выборке** (не участвовала в обучении).
Это даёт честную оценку качества модели на новых данных.

In [ ]:
rf_tuned_r2 = rf_search.best_score_
xgb_tuned_r2 = xgb_search.best_score_

if rf_tuned_r2 >= xgb_tuned_r2:
    best_model = rf_search.best_estimator_
    print(f'Лучшая модель: Random Forest (R² CV = {rf_tuned_r2:.4f})')
else:
    best_model = xgb_search.best_estimator_
    print(f'Лучшая модель: XGBoost (R² CV = {xgb_tuned_r2:.4f})')

best_preds = best_model.predict(X_test)
print(f'\nОценка на тестовой выборке:')
print(f'MAE:  {mean_absolute_error(y_test, best_preds):>12,.2f} руб.')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, best_preds)):>12,.2f} руб.')
print(f'R²:   {r2_score(y_test, best_preds):>12.4f}')

## 11. Анализ важности признаков

Для ансамблевых моделей (RF, XGBoost) доступен атрибут **feature_importances_**.
Он показывает, какие признаки модель считает наиболее важными при принятии решений.

**Два графика:**
1. **Топ-15 важных признаков** — горизонтальная столбчатая диаграмма
2. **Кумулятивная важность** — сколько признаков нужно для 80% и 95% важности

**Зачем:** определяем ключевые факторы ценообразования, можем отбросить неважные признаки.

In [ ]:
if hasattr(best_model.named_steps['model'], 'feature_importances_'):
    importances = best_model.named_steps['model'].feature_importances_
    fi_df = pd.DataFrame({
        'Feature': numeric_features,
        'Importance': importances
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    fi_df['Cumulative'] = fi_df['Importance'].cumsum()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(x='Importance', y='Feature', data=fi_df.head(15),
                palette='viridis', ax=axes[0])
    axes[0].set_title('Топ-15 важных признаков', fontsize=14)
    axes[1].plot(fi_df.index + 1, fi_df['Cumulative'] * 100,
                 marker='o', linewidth=2, markersize=6)
    axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% порог')
    axes[1].axhline(y=95, color='orange', linestyle='--', alpha=0.7, label='95% порог')
    axes[1].set_title('Кумулятивная важность признаков', fontsize=14)
    axes[1].set_xlabel('Количество признаков')
    axes[1].set_ylabel('Кумулятивная важность (%)')
    axes[1].legend()
    plt.tight_layout()
    pdf.savefig()
    plt.show()
    plt.close()

    n_80 = (fi_df['Cumulative'] <= 0.8).sum() + 1
    n_95 = (fi_df['Cumulative'] <= 0.95).sum() + 1
    print(f'Признаков для 80% важности: {n_80}')
    print(f'Признаков для 95% важности: {n_95}')
    fi_df

## 12. Интерпретация модели

Визуализируем качество предсказаний:
- **Реальные vs Предсказанные** — точки должны лежать вдоль диагонали (идеал — красная линия)
- **Распределение ошибок** — гистограмма разницы (y_true - y_pred). 
  Хорошо: ошибки распределены нормально вокруг нуля.
- **Q-Q plot (квантиль-квантиль)** — проверка нормальности распределения ошибок.

Также считаем **относительную ошибку** в процентах — более интерпретируема для бизнеса.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test, best_preds, alpha=0.4, edgecolors='k', linewidth=0.3)
min_val = min(y_test.min(), best_preds.min())
max_val = max(y_test.max(), best_preds.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Идеальная линия')
axes[0].set_xlabel('Реальные значения (руб.)')
axes[0].set_ylabel('Предсказанные значения (руб.)')
axes[0].set_title('Реальные vs Предсказанные цены')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

errors = y_test - best_preds
axes[1].hist(errors, bins=60, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Нулевая ошибка')
axes[1].axvline(x=errors.mean(), color='green', linestyle='--', linewidth=1.5,
                label=f'Средняя ошибка = {errors.mean():,.0f}')
axes[1].set_xlabel('Ошибка (руб.)'); axes[1].set_ylabel('Количество')
axes[1].set_title('Распределение ошибок модели')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

stats.probplot(errors, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q plot ошибок'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
pdf.savefig()
plt.show()
plt.close()

relative_errors = np.abs(errors) / y_test * 100
print(f'Средняя абсолютная относительная ошибка:  {relative_errors.mean():.2f}%')
print(f'Медианная абсолютная относительная ошибка: {relative_errors.median():.2f}%')
print(f'Доля предсказаний с ошибкой < 10%:         {(relative_errors < 10).mean() * 100:.1f}%')
print(f'Доля предсказаний с ошибкой < 20%:         {(relative_errors < 20).mean() * 100:.1f}%')

## 13. Кросс-валидация и сохранение модели

**5-fold кросс-валидация:** модель обучается на 4/5 данных, проверяется на 1/5 — и так 5 раз.
Это более надёжная оценка, чем single train/test split.

**Сохранение:**
- `best_model.pkl` — сериализованная лучшая модель (можно загрузить и использовать без переобучения)
- `eda_results.pdf` — все построенные графики в одном файле

In [ ]:
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f'R² по фолдам: {[f"{s:.4f}" for s in cv_scores]}')
print(f'Средний R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

joblib.dump(best_model, 'best_model.pkl')
print('\nМодель сохранена: best_model.pkl')
print('Графики сохранены: eda_results.pdf')

pdf.close()

## 14. Итоговые выводы

Сводка всех этапов проекта и ключевых результатов.

In [ ]:
print('=' * 70)
print('ИТОГОВЫЕ ВЫВОДЫ ПО ПРОЕКТУ')
print('=' * 70)
print(f'''
1. ЗАГРУЗКА И ОЧИСТКА
   - Исходный датасет: 22 676 записей, 12 признаков
   - Удалено дубликатов: {dup_count}
   - Удалено аномалий: {anomalies_before - len(df)} записей
   - Обработаны выбросы методом IQR (Winsorize) для 8 числовых признаков
   - Пропусков в данных не обнаружено

2. FEATURE ENGINEERING
   - Создано 14 новых признаков (Price_per_m2, Living_ratio, Renovation_score и др.)
   - Итоговое количество признаков: 26

3. ОТБОР ПРИЗНАКОВ
   - SelectKBest (F-тест): топ-3 — Area, Living area, Number of rooms

4. БАЗОВАЯ МОДЕЛЬ
   - Linear Regression: R² = {baseline_r2:.4f}, MAE = {baseline_mae:,.0f} руб.

5. СРАВНЕНИЕ МОДЕЛЕЙ
   - Обучено 8 моделей
   - Лучшая до тюнинга: Extra Trees (R² = 0.9295)

6. ПОДБОР ГИПЕРПАРАМЕТРОВ
   - RandomizedSearchCV для Random Forest и XGBoost
   - Победитель: {'XGBoost' if xgb_tuned_r2 >= rf_tuned_r2 else 'Random Forest'}
   - R² на тесте: {r2_score(y_test, best_preds):.4f}

7. ИНТЕРПРЕТИРУЕМОСТЬ
   - Средняя относительная ошибка: {relative_errors.mean():.2f}%
   - Медианная относительная ошибка: {relative_errors.median():.2f}%
   - Доля предсказаний с ошибкой < 10%: {(relative_errors < 10).mean() * 100:.1f}%
   - Доля предсказаний с ошибкой < 20%: {(relative_errors < 20).mean() * 100:.1f}%

8. КРОСС-ВАЛИДАЦИЯ
   - 5-fold CV: средний R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}

9. КЛЮЧЕВЫЕ ПРИЗНАКИ (определяющие цену):
   - Площадь (Area) и жилая площадь (Living area)
   - Тип ремонта (Renovation_score)
   - Количество комнат (Number of rooms)
   - Регион (Moscow vs область)
''')
print('=' * 70)
print('ПРОЕКТ ВЫПОЛНЕН')
print('=' * 70)